# 株価相関ネットワークとMST (Stock Correlation Network & Minimum Spanning Tree)

## 導入

金融市場において、個々の株価は独立に動くのではなく、マクロ経済要因・セクター特性・企業間の取引関係などを通じて相互に関連しています。
こうした株価間の **相関構造をネットワークとして表現** し、その中から **最小全域木（Minimum Spanning Tree, MST）** を抽出することで、
市場全体の階層的な構造を簡潔に可視化・分析する手法が、計量ファイナンスとネットワーク科学の交差領域で広く研究されています。

### 背景

- **Mantegna (1999)** は、株価の相関係数から距離指標を定義し、MSTを用いて米国株式市場の階層構造を明らかにしました。
- この手法は、同一セクターに属する銘柄が自然にクラスターを形成することを示し、市場のメゾ構造（中間規模の構造）を理解する強力なツールとなっています。
- MSTは $N$ 個の銘柄に対して $N-1$ 本のエッジのみを保持するため、情報のフィルタリングとしても機能します。

### このノートブックの目的

1. Yahoo Finance から日本の代表的な上場企業20社の株価データを取得する
2. 対数リターンに基づくピアソン相関行列を計算する
3. 相関を距離に変換し、MSTを構築する
4. ネットワークとして可視化し、中心性指標を算出する
5. 結果からマーケット構造に関する知見を得る

### 関連ドキュメント

- [01-overview.md](../01-overview.md) — 研究領域の全体像
- [06-quantitative-methods.md](../06-quantitative-methods.md) — 定量的手法の詳細（相関ネットワーク、MSTを含む）

## 環境セットアップ

必要なライブラリをインストールします。既にインストール済みの場合はスキップされます。

In [ ]:
# 必要なパッケージのインストール
!pip install networkx yfinance matplotlib pandas numpy scipy

## ライブラリのインポート

In [ ]:
import networkx as nx
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix

# 日本語フォントの設定（環境に応じて変更してください）
# macOS の場合
plt.rcParams['font.family'] = 'Hiragino Sans'
# Windows の場合は以下を使用:
# plt.rcParams['font.family'] = 'MS Gothic'
# Linux の場合は以下を使用:
# plt.rcParams['font.family'] = 'IPAGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['figure.dpi'] = 100

print("ライブラリのインポートが完了しました。")

## データ取得

Yahoo Finance から日本の代表的な上場企業20社の過去1年間の終値（Adjusted Close）を取得します。
各銘柄にはティッカーシンボルと日本語の表示名を設定します。

In [ ]:
# 銘柄の定義: ティッカーシンボル -> 日本語名
tickers_dict = {
    '7203.T': 'トヨタ',
    '6758.T': 'ソニー',
    '8306.T': '三菱UFJ',
    '9984.T': 'ソフトバンクG',
    '6861.T': 'キーエンス',
    '7974.T': '任天堂',
    '9432.T': 'NTT',
    '9433.T': 'KDDI',
    '6501.T': '日立',
    '6752.T': 'パナソニック',
    '9983.T': 'ファーストリテイリング',
    '8035.T': '東京エレクトロン',
    '4063.T': '信越化学',
    '6367.T': 'ダイキン',
    '6098.T': 'リクルート',
    '8316.T': '三井住友FG',
    '8411.T': 'みずほFG',
    '8001.T': '伊藤忠',
    '8058.T': '三菱商事',
    '8031.T': '三井物産',
}

# セクター分類の定義
sector_map = {
    'トヨタ': '自動車',
    'ソニー': 'テック',
    '三菱UFJ': '銀行',
    'ソフトバンクG': 'テック',
    'キーエンス': 'テック',
    '任天堂': 'テック',
    'NTT': '通信',
    'KDDI': '通信',
    '日立': 'テック',
    'パナソニック': 'テック',
    'ファーストリテイリング': 'その他',
    '東京エレクトロン': 'テック',
    '信越化学': 'その他',
    'ダイキン': 'その他',
    'リクルート': 'その他',
    '三井住友FG': '銀行',
    'みずほFG': '銀行',
    '伊藤忠': '商社',
    '三菱商事': '商社',
    '三井物産': '商社',
}

# セクターごとの色の定義
sector_colors = {
    '銀行': '#4169E1',      # 青
    '自動車': '#DC143C',    # 赤
    '通信': '#228B22',      # 緑
    '商社': '#FF8C00',      # オレンジ
    'テック': '#8A2BE2',    # 紫
    'その他': '#808080',    # グレー
}

# ティッカーシンボルのリスト
tickers = list(tickers_dict.keys())

print(f"取得対象: {len(tickers)} 銘柄")
for ticker, name in tickers_dict.items():
    sector = sector_map[name]
    print(f"  {ticker}: {name} ({sector})")

In [ ]:
# Yahoo Finance からデータを取得（過去1年間）
print("株価データをダウンロード中...")
data = yf.download(tickers, period='1y', auto_adjust=True)

# 終値（Close）を抽出
close_prices = data['Close']

# カラム名をティッカーから日本語名に変換
close_prices.columns = [tickers_dict[col] for col in close_prices.columns]

# 欠損値の確認と除去
print(f"\nデータ期間: {close_prices.index[0].strftime('%Y-%m-%d')} ~ {close_prices.index[-1].strftime('%Y-%m-%d')}")
print(f"取得日数: {len(close_prices)} 日")
print(f"欠損値の数:\n{close_prices.isnull().sum()}")

# 欠損値を前方補完で埋める
close_prices = close_prices.ffill().dropna()
print(f"\n欠損値処理後の取得日数: {len(close_prices)} 日")

# データの先頭を確認
close_prices.head()

## 対数リターンの計算

株価の変動率を対数リターン（log return）として計算します。
対数リターンは以下の式で定義されます:

$$r_i(t) = \ln P_i(t) - \ln P_i(t-1)$$

対数リターンを使用する理由:
- 時間的な加法性を持つ（複数期間のリターンを単純に足し合わせられる）
- 正規分布に近い分布を持つ傾向がある
- 小さな値の場合、単純リターンとほぼ等しい

In [ ]:
# 対数リターンの計算
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

print(f"対数リターンのデータ数: {len(log_returns)} 日")
print(f"銘柄数: {len(log_returns.columns)}")
print("\n--- 対数リターンの基本統計量 ---")
log_returns.describe().round(6)

## 相関行列の計算と可視化

対数リターン間のピアソン相関係数を計算します。
相関係数 $\rho_{ij}$ は銘柄 $i$ と銘柄 $j$ のリターンの線形関連性を測定します:

$$\rho_{ij} = \frac{\text{Cov}(r_i, r_j)}{\sigma_i \sigma_j}$$

ここで $\text{Cov}(r_i, r_j)$ は共分散、$\sigma_i, \sigma_j$ はそれぞれの標準偏差です。

In [ ]:
# ピアソン相関行列の計算
corr_matrix = log_returns.corr()

print("相関行列のサイズ:", corr_matrix.shape)
print(f"相関係数の範囲: {corr_matrix.min().min():.4f} ~ {corr_matrix.max().max():.4f}")
print(f"対角要素を除く平均相関: {corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].mean():.4f}")

# ヒートマップによる可視化
fig, ax = plt.subplots(figsize=(14, 12))

# ヒートマップの描画
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')

# 軸ラベルの設定
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(corr_matrix.columns, fontsize=10)

# 相関係数の値をセル内に表示
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        val = corr_matrix.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

# カラーバーの追加
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('ピアソン相関係数', fontsize=12)

ax.set_title('株価対数リターンの相関行列', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n--- 相関が最も高いペア（上位10組）---")
# 上三角行列の要素を取得してソート
corr_pairs = []
names = corr_matrix.columns.tolist()
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        corr_pairs.append((names[i], names[j], corr_matrix.values[i, j]))

corr_pairs.sort(key=lambda x: x[2], reverse=True)
for name_i, name_j, rho in corr_pairs[:10]:
    print(f"  {name_i} - {name_j}: {rho:.4f}")

## 距離行列の計算

相関係数を距離指標に変換します。Mantegna (1999) が提案した距離関数を使用します:

$$d(i,j) = \sqrt{2(1 - \rho_{ij})}$$

この距離は以下の性質を持ちます:
- $\rho_{ij} = 1$（完全正相関）のとき $d = 0$
- $\rho_{ij} = 0$（無相関）のとき $d = \sqrt{2} \approx 1.414$
- $\rho_{ij} = -1$（完全逆相関）のとき $d = 2$
- ユークリッド距離の公理（非負性、対称性、三角不等式）を全て満たす

In [ ]:
# 距離行列の計算: d(i,j) = sqrt(2 * (1 - rho(i,j)))
dist_matrix = np.sqrt(2 * (1 - corr_matrix.values))

# 数値誤差により対角要素が厳密に0でない場合があるため修正
np.fill_diagonal(dist_matrix, 0)

print(f"距離行列のサイズ: {dist_matrix.shape}")
print(f"距離の範囲（対角除く）: {dist_matrix[np.triu_indices_from(dist_matrix, k=1)].min():.4f} ~ {dist_matrix[np.triu_indices_from(dist_matrix, k=1)].max():.4f}")
print(f"平均距離: {dist_matrix[np.triu_indices_from(dist_matrix, k=1)].mean():.4f}")

# 距離行列をDataFrameとして表示
dist_df = pd.DataFrame(dist_matrix, index=corr_matrix.index, columns=corr_matrix.columns)
dist_df.round(3)

## 最小全域木（MST）の構築

距離行列から最小全域木（Minimum Spanning Tree）を構築します。
MSTは全てのノード（銘柄）を結ぶ木構造のうち、エッジの重み（距離）の合計が最小となるものです。

$N$ 個のノードに対して $N-1$ 本のエッジのみを保持するため、
完全グラフの $N(N-1)/2$ 本のエッジから本質的な関係のみを抽出するフィルタリングの役割を果たします。

In [ ]:
# scipy の minimum_spanning_tree を使用してMSTを構築
# 入力は疎行列（上三角部分）
dist_sparse = csr_matrix(np.triu(dist_matrix, k=1))
mst_sparse = minimum_spanning_tree(csr_matrix(dist_matrix))

# MST の結果を密行列に変換
mst_array = mst_sparse.toarray()

# networkx のグラフに変換
stock_names = corr_matrix.columns.tolist()
G = nx.Graph()

# ノードの追加（日本語名とセクター情報を属性として付与）
for name in stock_names:
    G.add_node(name, sector=sector_map[name])

# エッジの追加（MSTのエッジのみ）
for i in range(len(stock_names)):
    for j in range(len(stock_names)):
        if mst_array[i, j] > 0:
            # エッジの重みとして距離と相関係数の両方を保持
            G.add_edge(
                stock_names[i],
                stock_names[j],
                weight=mst_array[i, j],
                correlation=corr_matrix.values[i, j]
            )

print(f"MST の構築が完了しました。")
print(f"  ノード数: {G.number_of_nodes()}")
print(f"  エッジ数: {G.number_of_edges()}")
print(f"  木構造の検証（ノード数 - 1 = エッジ数）: {G.number_of_nodes() - 1 == G.number_of_edges()}")
print(f"  連結性の検証: {nx.is_connected(G)}")

# MST のエッジ情報を表示
print("\n--- MST のエッジ一覧（距離の昇順）---")
edges_info = []
for u, v, d in G.edges(data=True):
    edges_info.append((u, v, d['weight'], d['correlation']))

edges_info.sort(key=lambda x: x[2])
for u, v, dist, corr in edges_info:
    print(f"  {u} -- {v}: 距離={dist:.4f}, 相関={corr:.4f}")

## ネットワーク可視化

構築したMSTをネットワークとして可視化します。
- **ノードの色**: セクターごとに色分け
- **ノードのサイズ**: 次数（接続数）に比例
- **エッジの太さ**: 相関の強さに比例

In [ ]:
# ネットワーク可視化
fig, ax = plt.subplots(figsize=(16, 12))

# レイアウトの計算（Kamada-Kawai レイアウトは距離ベースのネットワークに適している）
pos = nx.kamada_kawai_layout(G, weight='weight')

# ノードの色をセクターに基づいて設定
node_colors = [sector_colors[sector_map[node]] for node in G.nodes()]

# ノードサイズを次数に比例させる（最小値を設定）
degrees = dict(G.degree())
min_size = 800
max_size = 3000
max_degree = max(degrees.values())
min_degree = min(degrees.values())
if max_degree > min_degree:
    node_sizes = [
        min_size + (max_size - min_size) * (degrees[node] - min_degree) / (max_degree - min_degree)
        for node in G.nodes()
    ]
else:
    node_sizes = [min_size + (max_size - min_size) / 2 for _ in G.nodes()]

# エッジの太さを相関の強さに比例させる
edge_widths = []
for u, v, d in G.edges(data=True):
    edge_widths.append(1 + 3 * abs(d['correlation']))

# ノードの描画
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=node_colors,
    node_size=node_sizes,
    edgecolors='black',
    linewidths=1.5,
    alpha=0.9
)

# エッジの描画
nx.draw_networkx_edges(
    G, pos, ax=ax,
    width=edge_widths,
    edge_color='#555555',
    alpha=0.7
)

# ラベルの描画
nx.draw_networkx_labels(
    G, pos, ax=ax,
    font_size=9,
    font_weight='bold',
    font_family='Hiragino Sans'
)

# エッジラベル（距離）の描画
edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(
    G, pos, ax=ax,
    edge_labels=edge_labels,
    font_size=7,
    font_color='#CC0000',
    font_family='Hiragino Sans'
)

# 凡例の作成
legend_handles = []
for sector, color in sector_colors.items():
    # そのセクターに属する銘柄が存在する場合のみ凡例に追加
    if any(sector_map[node] == sector for node in G.nodes()):
        legend_handles.append(
            plt.scatter([], [], c=color, s=150, edgecolors='black', linewidths=1, label=sector)
        )

ax.legend(
    handles=legend_handles,
    loc='upper left',
    fontsize=11,
    title='セクター',
    title_fontsize=12,
    framealpha=0.9
)

ax.set_title('株価相関ネットワークの最小全域木（MST）', fontsize=18, fontweight='bold', pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

## ネットワーク指標の計算

MSTの構造的特徴を定量化するため、以下の中心性指標を計算します:

- **次数中心性（Degree Centrality）**: ノードに接続するエッジの数。MSTにおいて次数が高いノードは「ハブ」として多くの銘柄を繋ぐ役割を果たす。
- **媒介中心性（Betweenness Centrality）**: ネットワーク内の最短経路のうち、そのノードを経由する割合。値が高いノードは情報（影響）の伝播において重要な仲介者。
- **近接中心性（Closeness Centrality）**: 他の全ノードへの最短距離の逆数の平均。値が高いノードはネットワークの中心に位置し、全体への影響力が大きい。

In [ ]:
# 中心性指標の計算
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
closeness_centrality = nx.closeness_centrality(G)

# 結果をDataFrameにまとめる
centrality_df = pd.DataFrame({
    '銘柄': list(G.nodes()),
    'セクター': [sector_map[node] for node in G.nodes()],
    '次数': [G.degree(node) for node in G.nodes()],
    '次数中心性': [degree_centrality[node] for node in G.nodes()],
    '媒介中心性': [betweenness_centrality[node] for node in G.nodes()],
    '近接中心性': [closeness_centrality[node] for node in G.nodes()],
})

# 次数中心性の降順でソート
centrality_df = centrality_df.sort_values('次数中心性', ascending=False).reset_index(drop=True)

print("=" * 80)
print("ネットワーク中心性指標")
print("=" * 80)
print(centrality_df.to_string(index=False, float_format='{:.4f}'.format))

# ハブノード（次数が最大のノード）の特定
max_degree_node = centrality_df.iloc[0]['銘柄']
max_betweenness_node = centrality_df.sort_values('媒介中心性', ascending=False).iloc[0]['銘柄']
max_closeness_node = centrality_df.sort_values('近接中心性', ascending=False).iloc[0]['銘柄']

print(f"\n--- 各指標で最も中心的な銘柄 ---")
print(f"  次数中心性が最大: {max_degree_node} ({sector_map[max_degree_node]})")
print(f"  媒介中心性が最大: {max_betweenness_node} ({sector_map[max_betweenness_node]})")
print(f"  近接中心性が最大: {max_closeness_node} ({sector_map[max_closeness_node]})")

In [ ]:
# 中心性指標の棒グラフ可視化
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

metrics = [
    ('次数中心性', '次数中心性'),
    ('媒介中心性', '媒介中心性'),
    ('近接中心性', '近接中心性'),
]

for ax, (col, title) in zip(axes, metrics):
    # 値の降順でソート
    sorted_df = centrality_df.sort_values(col, ascending=True)
    colors = [sector_colors[sector_map[name]] for name in sorted_df['銘柄']]

    ax.barh(sorted_df['銘柄'], sorted_df[col], color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(col, fontsize=11)
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('MST ネットワーク中心性指標の比較', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 考察

### 結果の解釈

#### セクタークラスターの形成

MSTの可視化結果から、**同一セクターに属する銘柄が隣接してクラスターを形成する傾向** が確認できます。
これは以下のことを示唆しています:

- **銀行セクター**（三菱UFJ、三井住友FG、みずほFG）: 金利動向や金融政策に同様に反応するため、高い相関を持つ
- **商社セクター**（伊藤忠、三菱商事、三井物産）: 資源価格や為替レートなど共通の要因に影響される
- **通信セクター**（NTT、KDDI）: 規制環境やディフェンシブ銘柄としての特性を共有
- **テックセクター**: より多様であるが、半導体関連（東京エレクトロン、信越化学）などのサブクラスターが形成される可能性がある

#### ハブ銘柄の意味

MST上で多くの銘柄と接続する「ハブ」銘柄は、市場全体の動きを媒介する役割を果たしています。
媒介中心性が高い銘柄は、異なるセクター間の「ブリッジ」として機能し、
市場のシステミックリスクの伝播経路を理解する上で重要です。

### 金融危機時のMST構造変化

先行研究では、金融危機時にMSTの構造が大きく変化することが報告されています:

- **Onnela et al. (2003)**: 危機時にはMSTが「縮約」し、平均距離が短くなる（全銘柄が同方向に動く）
- **Boginski et al. (2005)**: 閾値ネットワークのクリーク構造が危機時に変化する
- **日本市場**: バブル崩壊、リーマンショック、コロナショックなどでMST構造の変化が観察されている

### 発展的な話題

本ノートブックで扱ったMSTは最も基本的なフィルタリング手法ですが、以下のような発展的手法も存在します:

1. **閾値ネットワーク（Threshold Network）**: 相関が一定値以上のエッジのみを保持する方法。閾値の選択が結果に大きく影響する。

2. **時間窓分析（Rolling Window Analysis）**: 時間窓をスライドさせながらMSTを逐次構築し、市場構造のダイナミクスを追跡する。Survival ratio（連続する時間窓間でのエッジの生存率）などの指標が使われる。

3. **PMFG（Planar Maximally Filtered Graph）**: Tumminello et al. (2005) が提案。MSTの $N-1$ 本に対して $3(N-2)$ 本のエッジを保持でき、より豊富な構造情報を抽出できる。平面グラフの制約の下で最大限のエッジを含む。

4. **部分相関・偏相関ネットワーク**: 第三の変数の影響を除いた直接的な関連性のみを抽出するアプローチ。

5. **動的条件付き相関（DCC）モデル**: 時変相関を推定するGARCH系モデルと組み合わせた手法。

これらの手法の詳細については [06-quantitative-methods.md](../06-quantitative-methods.md) を参照してください。